# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [24]:
# Load the data with proper encoding
df = pd.read_csv('AviationData.csv', encoding='latin-1')

# Inspect the dataframe
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nBasic statistics:")
print(df.describe())

C:\Users\HP\AppData\Local\Temp\ipykernel_12032\1642802256.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('AviationData.csv', encoding='latin-1')


Dataset shape: (88889, 31)

First few rows:
         Event.Id Investigation.Type Accident.Number  Event.Date  \
0  20001218X45444           Accident      SEA87LA080  1948-10-24   
1  20001218X45447           Accident      LAX94LA336  1962-07-19   
2  20061025X01555           Accident      NYC07LA005  1974-08-30   
3  20001218X45448           Accident      LAX96LA321  1977-06-19   
4  20041105X01764           Accident      CHI79FA064  1979-08-02   

          Location        Country   Latitude  Longitude Airport.Code  \
0  MOOSE CREEK, ID  United States        NaN        NaN          NaN   
1   BRIDGEPORT, CA  United States        NaN        NaN          NaN   
2    Saltville, VA  United States  36.922223 -81.878056          NaN   
3       EUREKA, CA  United States        NaN        NaN          NaN   
4       Canton, OH  United States        NaN        NaN          NaN   

  Airport.Name  ... Purpose.of.flight Air.carrier Total.Fatal.Injuries  \
0          NaN  ...          Personal   

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [25]:
# Extract year from Event.Date and filter for 1983 onwards
# First, convert to datetime
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df['Year'] = df['Event.Date'].dt.year

# Filter for events from 1983 onwards
df = df[df['Year'] >= 1983].copy()

print(f"Dataset after filtering for 1983 onwards: {df.shape}")
print(f"Year range: {df['Year'].min()} to {df['Year'].max()}")

# Check aircraft categories
print("\nAircraft Categories:")
print(df['Aircraft.Category'].value_counts())

Dataset after filtering for 1983 onwards: (85289, 32)
Year range: 1983 to 2022

Aircraft Categories:
Aircraft.Category
Airplane             24441
Helicopter            3151
Glider                 456
Balloon                201
Weight-Shift           161
Gyrocraft              158
Powered Parachute       91
Ultralight              29
Unknown                 13
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [26]:
# Filter to include only airplanes (not helicopters, balloons, etc.)
df = df[df['Aircraft.Category'] == 'Airplane'].copy()
print(f"Dataset after filtering for Airplanes: {df.shape}")

# Examine injury-related columns
print("\nInjury columns sample:")
print(df[['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']].head(10))

# Create a metric for passenger count (estimate as sum of all injury categories + uninjured)
df['Estimated.Passengers'] = (df['Total.Fatal.Injuries'].fillna(0) + 
                               df['Total.Serious.Injuries'].fillna(0) + 
                               df['Total.Minor.Injuries'].fillna(0) + 
                               df['Total.Uninjured'].fillna(0))

# Create fatal/serious injury metric: fraction of passengers with fatal or serious injuries
df['Serious.Fatal.Injury.Count'] = (df['Total.Fatal.Injuries'].fillna(0) + 
                                     df['Total.Serious.Injuries'].fillna(0))

# Only calculate fraction for flights with passengers
df['Serious.Fatal.Injury.Fraction'] = 0.0
mask = df['Estimated.Passengers'] > 0
df.loc[mask, 'Serious.Fatal.Injury.Fraction'] = (
    df.loc[mask, 'Serious.Fatal.Injury.Count'] / df.loc[mask, 'Estimated.Passengers']
)

print("\nInjury metrics created:")
print(df[['Estimated.Passengers', 'Serious.Fatal.Injury.Count', 'Serious.Fatal.Injury.Fraction']].describe())

Dataset after filtering for Airplanes: (24441, 32)

Injury columns sample:
      Total.Fatal.Injuries  Total.Serious.Injuries  Total.Minor.Injuries  \
4149                   NaN                     NaN                   NaN   
4150                   NaN                     NaN                   NaN   
4171                   1.0                     1.0                   NaN   
4285                   1.0                     NaN                   NaN   
5957                   NaN                     NaN                   NaN   
5960                  11.0                     2.0                   NaN   
6669                   1.0                     NaN                   NaN   
6760                   NaN                     NaN                   NaN   
6806                   NaN                     NaN                   NaN   
7084                   NaN                     NaN                   NaN   

      Total.Uninjured  
4149            588.0  
4150            588.0  
4171            

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [27]:
# Examine Aircraft.Damage column
print("Aircraft.Damage value counts:")
print(df['Aircraft.damage'].value_counts(dropna=False))

# Create a binary indicator for whether aircraft was destroyed
# "Destroyed" = 1, otherwise = 0
df['Aircraft.Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

print("\nAircraft.Destroyed value counts:")
print(df['Aircraft.Destroyed'].value_counts())
print(f"\nFraction destroyed: {df['Aircraft.Destroyed'].mean():.3f}")

Aircraft.Damage value counts:
Aircraft.damage
Substantial    19612
Destroyed       2637
NaN             1238
Minor            854
Unknown          100
Name: count, dtype: int64

Aircraft.Destroyed value counts:
Aircraft.Destroyed
0    21804
1     2637
Name: count, dtype: int64

Fraction destroyed: 0.108


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [28]:
# Investigate the Make column
print("Number of unique makes:", df['Make'].nunique())
print("\nMake value counts (top 30):")
print(df['Make'].value_counts().head(30))
print("\nNumber of NaN in Make:", df['Make'].isna().sum())

# List of cleaning tasks for Make column:
# 1. Remove rows with missing Make values
# 2. Keep only makes with at least 50 accidents
# 3. Strip whitespace

print("\n--- Cleaning tasks for Make column ---")
print("1. Remove rows with missing Make values")
print("2. Keep only makes with at least 50 accidents")
print("3. Strip whitespace")

# Execute cleaning
df = df[df['Make'].notna()].copy()
print(f"\nAfter removing NaN Makes: {df.shape}")

# Count by make
make_counts = df['Make'].value_counts()
makes_to_keep = make_counts[make_counts >= 50].index.tolist()
print(f"Makes with >= 50 accidents: {len(makes_to_keep)}")

df = df[df['Make'].isin(makes_to_keep)].copy()
print(f"After filtering for makes with >= 50 accidents: {df.shape}")

# Strip whitespace
df['Make'] = df['Make'].str.strip()

print(f"\nMake column cleaned. Unique makes: {df['Make'].nunique()}")

Number of unique makes: 3742

Make value counts (top 30):
Make
CESSNA                      4867
PIPER                       2805
Cessna                      2283
Piper                       1189
BOEING                      1037
BEECH                       1018
Beech                        416
Boeing                       242
MOONEY                       238
CIRRUS DESIGN CORP           218
AIR TRACTOR INC              217
AIRBUS                       216
BELLANCA                     158
AERONCA                      149
MAULE                        144
Mooney                       125
EMBRAER                      123
Air Tractor                  117
LUSCOMBE                      95
STINSON                       91
CHAMPION                      91
DEHAVILLAND                   91
AIR TRACTOR                   89
CIRRUS                        80
NORTH AMERICAN                79
GRUMMAN                       78
DIAMOND AIRCRAFT IND INC      74
AVIAT AIRCRAFT INC            72
Maule        

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [29]:
# Clean and standardize Model column
# Remove NaN values
df = df[df['Model'].notna()].copy()
print(f"After removing NaN Models: {df.shape}")

# Standardize Model to uppercase
df['Model'] = df['Model'].str.upper().str.strip()

# Create a unique identifier combining Make and Model
df['Aircraft.Type'] = df['Make'] + ' ' + df['Model']

print(f"\nUnique aircraft types: {df['Aircraft.Type'].nunique()}")
print("\nTop aircraft types:")
print(df['Aircraft.Type'].value_counts().head(20))

After removing NaN Models: (17380, 36)

Unique aircraft types: 2587

Top aircraft types:
Aircraft.Type
CESSNA 172                 493
BOEING 737                 368
Cessna 172                 276
CESSNA 152                 193
CESSNA 182                 193
CESSNA 172S                185
PIPER PA28                 179
CESSNA 172N                161
CIRRUS DESIGN CORP SR22    143
PIPER PA-18-150            134
CESSNA 172M                131
CESSNA 180                 127
BEECH A36                  124
Cessna 152                 123
PIPER PA-28-140            120
CESSNA 150                 111
Cessna 182                 111
Piper PA28                  94
Cessna 172S                 91
CESSNA 172P                 91
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [30]:
# Inspect and clean other relevant columns

print("=== Engine.Type ===")
print(f"Missing: {df['Engine.Type'].isna().sum()}")
print(f"Unique values: {df['Engine.Type'].nunique()}")
print(df['Engine.Type'].value_counts())
df['Engine.Type'] = df['Engine.Type'].str.upper().str.strip()

print("\n=== Weather.Condition ===")
print(f"Missing: {df['Weather.Condition'].isna().sum()}")
print(f"Unique values: {df['Weather.Condition'].nunique()}")
print(df['Weather.Condition'].value_counts())
df['Weather.Condition'] = df['Weather.Condition'].str.upper().str.strip()

print("\n=== Number.of.Engines ===")
print(f"Missing: {df['Number.of.Engines'].isna().sum()}")
print(f"Data type: {df['Number.of.Engines'].dtype}")
print(df['Number.of.Engines'].value_counts().sort_index())
# Convert to numeric
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

print("\n=== Purpose.of.flight ===")
print(f"Missing: {df['Purpose.of.flight'].isna().sum()}")
print(f"Unique values: {df['Purpose.of.flight'].nunique()}")
print(df['Purpose.of.flight'].value_counts())
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.upper().str.strip()

print("\n=== Broad.phase.of.flight ===")
print(f"Missing: {df['Broad.phase.of.flight'].isna().sum()}")
print(f"Unique values: {df['Broad.phase.of.flight'].nunique()}")
print(df['Broad.phase.of.flight'].value_counts())
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.upper().str.strip()

=== Engine.Type ===
Missing: 3166
Unique values: 8
Engine.Type
Reciprocating      12562
Turbo Prop           839
Turbo Fan            640
Unknown               94
Turbo Jet             59
Geared Turbofan       11
Turbo Shaft            8
UNK                    1
Name: count, dtype: int64

=== Weather.Condition ===
Missing: 2360
Unique values: 4
Weather.Condition
VMC    13898
IMC      880
Unk      180
UNK       62
Name: count, dtype: int64

=== Number.of.Engines ===
Missing: 2036
Data type: float64
Number.of.Engines
0.0        4
1.0    12906
2.0     2344
3.0       23
4.0       67
Name: count, dtype: int64

=== Purpose.of.flight ===
Missing: 2893
Unique values: 24
Purpose.of.flight
Personal                     9650
Instructional                2376
Aerial Application            678
Business                      398
Unknown                       286
Positioning                   256
Skydiving                     148
Aerial Observation            142
Other Work Use                114
Banne

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [31]:
# Examine NaN counts and percentages
print("Columns with NaN values and their percentages:")
nan_info = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df)) * 100
})
nan_info = nan_info[nan_info['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(nan_info)

# Drop only columns that exist and have >50% missing
threshold = 0.5
cols_to_drop = nan_info[nan_info['Missing %'] > threshold * 100].index.tolist()
print(f"\nColumns to drop (>50% missing): {cols_to_drop}")

# Only drop columns that actually exist in the dataframe
existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols_to_drop)

print(f"\nDataset shape after dropping columns: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Columns with NaN values and their percentages:
                        Missing Count  Missing %
Schedule                        15384  88.515535
Broad.phase.of.flight           15107  86.921749
Air.carrier                      9171  52.767549
Airport.Code                     6020  34.637514
Airport.Name                     5922  34.073648
Report.Status                    3725  21.432681
Engine.Type                      3166  18.216341
Purpose.of.flight                2893  16.645570
Weather.Condition                2360  13.578826
Total.Serious.Injuries           2288  13.164557
Total.Fatal.Injuries             2214  12.738780
Total.Minor.Injuries             2060  11.852704
Number.of.Engines                2036  11.714614
Longitude                        1821  10.477560
Latitude                         1819  10.466053
Aircraft.damage                  1000   5.753740
Publication.Date                  768   4.418872
Injury.Severity                   690   3.970081
Total.Uninjured       

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [32]:
# Save the cleaned data to CSV
df.to_csv('data/AviationData_cleaned.csv', index=False)
print("Cleaned data saved to data/AviationData_cleaned.csv")
print(f"Final dataset shape: {df.shape}")
print("\nFinal dataset columns:")
print(df.columns.tolist())

Cleaned data saved to data/AviationData_cleaned.csv
Final dataset shape: (17380, 34)

Final dataset columns:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Report.Status', 'Publication.Date', 'Year', 'Estimated.Passengers', 'Serious.Fatal.Injury.Count', 'Serious.Fatal.Injury.Fraction', 'Aircraft.Destroyed', 'Aircraft.Type']
